In [ ]:
# ===============================
# 1. Import Libraries
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score

# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv("/content/oil data - Final_dataset.csv")

print("Columns:", df.columns)

# ===============================
# 3. Clean Columns
# ===============================
# Remove unwanted columns safely
df = df.drop(columns=["F6", "F12"], errors='ignore')

# Clean Oil_Type (IMPORTANT FIX)
df["Type_of_Oil"] = df["Type_of_Oil"].astype(str).str.strip().str.lower()

# ===============================
# 4. Prepare Features & Targets
# ===============================
X = df.drop(columns=["Adulteration %", "Type_of_Oil"])
y_reg = df["Adulteration %"]
y_clf = df["Type_of_Oil"]

# Ensure ONLY numeric data in X
X = X.select_dtypes(include=[np.number])

# ===============================
# 5. Train-Test Split
# ===============================
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# ===============================
# 6. Train Models
# ===============================
reg_model = RandomForestRegressor(n_estimators=150, random_state=42)
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)

reg_model.fit(X_train, y_reg_train)
clf_model.fit(X_train, y_clf_train)

# ===============================
# 7. Evaluation
# ===============================
# Regression
y_reg_pred = reg_model.predict(X_test)
mae = mean_absolute_error(y_reg_test, y_reg_pred)

# Classification
y_clf_pred = clf_model.predict(X_test)
acc = accuracy_score(y_clf_test, y_clf_pred)

print("\n===== RESULTS ===holz")
print("MAE (Adulteration % error):", round(mae, 2))
print("Oil Type Accuracy:", round(acc * 100, 2), "%")

# ===============================
# 8. Categorization Function
# ===============================
def categorize(value):
    if value <= 5:
        return "Pure"
    elif value <= 30:
        return "Mild Adulteration"
    elif value <= 60:
        return "High Adulteration"
    else:
        return "Very High Adulteration"

# ===============================
# 9. Test Sample Prediction
# ===============================
sample = X.sample(1)

pred_percent = reg_model.predict(sample)[0]
pred_type = clf_model.predict(sample)[0]
pred_category = categorize(pred_percent)

print("\n===== SAMPLE OUTPUT =====")
print("Oil Type:", pred_type)
print("Adulteration %:", round(pred_percent, 2))
print("Category:", pred_category)

from sklearn.model_selection import cross_val_score

scores = cross_val_score(reg_model, X, y_reg, cv=5, scoring='neg_mean_absolute_error')
print("Avg MAE:", -scores.mean())

Columns: Index(['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11',
       'F12', 'F13', 'F14', 'Adulteration %', 'Type_of_Oil'],
      dtype='object')

===== RESULTS ===holz
MAE (Adulteration % error): 0.16
Oil Type Accuracy: 100.0 %

===== SAMPLE OUTPUT =====
Oil Type: sunflower
Adulteration %: 80.0
Category: Very High Adulteration
Avg MAE: 10.740110177404294


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(reg_model, X, y_reg, cv=5, scoring='neg_mean_absolute_error')
print("Avg MAE:", -scores.mean())

Avg MAE: 10.740110177404294


In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_reg_test, y_reg_pred)
print("R2 Score:", r2)

R2 Score: 0.9996205332109904


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_clf_test, y_clf_pred))

              precision    recall  f1-score   support

   groundnut       1.00      1.00      1.00        29
   safflower       1.00      1.00      1.00        28
      sesame       1.00      1.00      1.00        30
   sunflower       1.00      1.00      1.00        33

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120



55.0, 144.0, 124.0, 7.0, 156.0, 17.0, 44.0, 85.0, 93.0, 156.0, 6.0, 38.0

In [ ]:
import numpy as np

sample = np.array([[41.0, 113.0, 97.0, 6.0, 119.0, 14.0, 34.0, 66.0, 76.0, 120.0, 6.0, 32.0]])

pred_percent = reg_model.predict(sample)[0]

print("Oil Type:", pred_type)
print("Adulteration %:", round(pred_percent, 2))

Oil Type: groundnut
Adulteration %: 60.27


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [ ]:
pip install skl2onnx onnx

In [ ]:
import pickle

# Save everything together
model_data = {
    "reg_model": reg_model,
    "clf_model": clf_model,
    "features": X.columns.tolist()
}

with open("model.pkl", "wb") as f:
    pickle.dump(model_data, f)

In [ ]:
!pip install onnx onnxruntime skl2onnx

In [ ]:
import onnx
import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Example: your trained model
model =  reg_model # replace this

# Define input shape (important!)
initial_type = [('float_input', FloatTensorType([None, X.shape[1]]))]

onnx_model = convert_sklearn(model, initial_types=initial_type)

# Save model
with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [ ]:
import numpy as np
import onnxruntime as rt

# Load model
sess = rt.InferenceSession("model.onnx")

input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

# Your input (IMPORTANT: 2D shape + float32)
sample = np.array([[57.0, 141.0, 120.0, 6.0, 152.0, 18.0, 45.0, 83.0, 89.0, 151.0, 6.0, 37.0]], dtype=np.float32)

# Run prediction
pred = sess.run([output_name], {input_name: sample})

print("Prediction:", pred[0][0])

Prediction: [70.266685]


In [ ]:
# ===============================
# EVALUATION CELL (RUN SEPARATELY)
# ===============================

import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import cross_val_score

# ===============================
# 1. Predictions
# ===============================
y_reg_pred = reg_model.predict(X_test)
y_clf_pred = clf_model.predict(X_test)

# ===============================
# 2. Regression Metrics
# ===============================
mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)

# MAPE (handle divide-by-zero safely)
mape = np.mean(np.abs((y_reg_test - y_reg_pred) / (y_reg_test + 1e-8))) * 100

print("\n===== REGRESSION METRICS =====")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 3))
print("MAPE:", round(mape, 2), "%")

# ===============================
# 3. Classification Metrics
# ===============================
acc = accuracy_score(y_clf_test, y_clf_pred)

print("\n===== CLASSIFICATION METRICS =====")
print("Accuracy:", round(acc * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_clf_test, y_clf_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_clf_test, y_clf_pred))

# ===============================
# 4. Cross Validation
# ===============================
mae_scores = cross_val_score(reg_model, X, y_reg, cv=5, scoring='neg_mean_absolute_error')
r2_scores = cross_val_score(reg_model, X, y_reg, cv=5, scoring='r2')

print("\n===== CROSS VALIDATION =====")
print("Avg MAE:", round(-mae_scores.mean(), 2))
print("Avg R2:", round(r2_scores.mean(), 3))


===== REGRESSION METRICS =====
MAE: 0.16
RMSE: 0.53
R2 Score: 1.0
MAPE: 166666667.24 %

===== CLASSIFICATION METRICS =====
Accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

   groundnut       1.00      1.00      1.00        29
   safflower       1.00      1.00      1.00        28
      sesame       1.00      1.00      1.00        30
   sunflower       1.00      1.00      1.00        33

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120

Confusion Matrix:
[[29  0  0  0]
 [ 0 28  0  0]
 [ 0  0 30  0]
 [ 0  0  0 33]]

===== CROSS VALIDATION =====
Avg MAE: 10.74
Avg R2: 0.648


In [ ]:
# ===== FIX ONNX MODEL VERSION =====

import onnx

# 🔹 Load your existing model (change name if needed)
model = onnx.load("model.onnx")

# 🔥 Force compatible IR version
model.ir_version = 9

# 🔹 Save fixed model
onnx.save(model, "model_fixed.onnx")

print("✅ Model fixed and saved as model_fixed.onnx")

✅ Model fixed and saved as model_fixed.onnx


In [ ]:
import onnx
from onnx import version_converter

# Load your model
model = onnx.load("model.onnx")

# 🔥 Convert opset to 15 (compatible)
converted_model = version_converter.convert_version(model, 15)

# Save
onnx.save(converted_model, "model_final.onnx")

print("✅ Converted to opset 15")

✅ Converted to opset 15


In [ ]:
import joblib

model = joblib.load("model.pkl")

print("✅ Model loaded")

✅ Model loaded


In [ ]:
!pip install skl2onnx

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# 🔥 adjust feature count (you used 12 earlier)
initial_type = [('input', FloatTensorType([None, 12]))]

onnx_model = convert_sklearn(
    model,
    initial_types=initial_type,
    target_opset=15
)

with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("✅ ONNX model created")

MissingShapeCalculator: Unable to find a shape calculator for type '<class 'dict'>'.
It usually means the pipeline being converted contains a
transformer or a predictor with no corresponding converter
implemented in sklearn-onnx. If the converted is implemented
in another library, you need to register
the converted so that it can be used by sklearn-onnx (function
update_registered_converter). If the model is not yet covered
by sklearn-onnx, you may raise an issue to
https://github.com/onnx/sklearn-onnx/issues
to get the converter implemented or even contribute to the
project. If the model is a custom model, a new converter must
be implemented. Examples can be found in the gallery.


In [ ]:
model.predict(X_test)